## Analisis de data de ventas

In [2]:
import pandas as pd

# Cargar el CSV
df = pd.read_csv('data/process/catusita_consolidated.csv')

# Imprimir las columnas para verificar su nombre
print("Columnas del CSV:", df.columns)

# Agrupar las ventas por artículo
# Cambia 'articulo' y 'dolares' por los nombres reales de tus columnas si son diferentes
ventas_por_articulo = df.groupby('articulo')['venta_usd'].sum().reset_index()

# Ordenar los resultados de mayor a menor según las ventas en dólares
ventas_por_articulo = ventas_por_articulo.sort_values('venta_usd', ascending=False)

# Obtener el artículo más vendido en dólares
articulo_mas_vendido = ventas_por_articulo.iloc[0]

print("El artículo más vendido en dólares es:")
print(articulo_mas_vendido)

Columnas del CSV: Index(['fecha', 'documento', 'articulo', 'cantidad', 'transacciones',
       'venta_pen', 'venta_usd', 'fuente_suministro', 'costo', 'lt'],
      dtype='object')
El artículo más vendido en dólares es:
articulo      paw68a500
venta_usd    1995489.28
Name: 5733, dtype: object


# Analisis de predicciones

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Lasso
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

class Predictor:
    def __init__(self):
        self.scaler = StandardScaler()

    def load_and_preprocess_data(self, data_path):
        df = pd.read_csv(data_path)
        df['fecha'] = pd.to_datetime(df['fecha'])
        df = df.sort_values(by=['articulo', 'fecha'])
        return df

    def load_covariates_data(self, cov_path):
        return pd.read_csv(cov_path)

    def create_monthly_sales_data(self, df):
        df.loc[:, 'year'] = df['fecha'].dt.year
        df.loc[:, 'month'] = df['fecha'].dt.month

        monthly_data = df.groupby(['articulo', 'year', 'month']).agg({
            'cantidad': 'sum',
            'transacciones': 'sum',
            'venta_pen': 'sum',
            'fuente_suministro': 'first',
            'lt': 'first'
        }).reset_index()

        monthly_data['fecha'] = pd.to_datetime(monthly_data.apply(
            lambda row: pd.Timestamp(year=int(row['year']), month=int(row['month']), day=1), axis=1))
        monthly_data['lt'] = monthly_data['lt'].fillna(0)

        hoy = pd.Timestamp.today()
        primer_dia_mes_actual = pd.Timestamp(hoy.year, hoy.month, 1)
        fecha_limite = primer_dia_mes_actual - pd.DateOffset(months=1)

        monthly_data = monthly_data[monthly_data['fecha'] <= fecha_limite]

        return monthly_data.sort_values(by=['articulo', 'fecha'])

    def select_features_with_lasso(self, X, y, feature_columns, alpha=0.01):
        """
        Perform feature selection using LASSO regression.
        Returns selected feature names based on non-zero coefficients.
        """
        X_scaled = self.scaler.fit_transform(X)
        
        lasso = Lasso(alpha=alpha, random_state=42)
        lasso.fit(X_scaled, y)
        
        selected_features = [feature for feature, coef in zip(feature_columns, lasso.coef_) 
                        if abs(coef) > 0]
                
        return selected_features

    def prepare_features_for_ml(self, all_monthly_data, df_cov, df_correlaciones_sig, sku):
        sku_correlations = df_correlaciones_sig[df_correlaciones_sig['sku'] == sku]
        
        if len(sku_correlations) == 0:
            return all_monthly_data[all_monthly_data['articulo'] == sku].copy(), []
            
        data = all_monthly_data[all_monthly_data['articulo'] == sku].merge(
            df_cov, on=['year', 'month'], how='left'
        )
        
        feature_columns = []
        for _, row in sku_correlations.iterrows():
            col_name = row['tipo']
            lag = row['lag']
            if lag > 0:
                data[f'{col_name}_lag_{lag}'] = data[col_name].shift(lag)
                feature_columns.append(f'{col_name}_lag_{lag}')
            else:
                feature_columns.append(col_name)
        
        data = data.dropna(axis=1, how='all')
        data = data.fillna(0)
        start_col = data.columns.get_loc("fecha")
        feature_columns = data.iloc[:, start_col + 1:].columns
        feature_columns = feature_columns.tolist()
        
        if len(data) > 0 and len(feature_columns) > 0:
            X = data[feature_columns]
            y = data['cantidad']
            selected_features = self.select_features_with_lasso(X, y, feature_columns)
            return data, selected_features
        
        return data, []

    def weighted_mape(self, y_true, y_pred):
        errors = []
        for true, pred in zip(y_true, y_pred):
            if pd.isna(true) or pd.isna(pred) or true == 0:
                continue
            error = abs((true - pred) / true)
            if pred < true:  # underprediction
                error *= 2
            errors.append(error)
        return np.mean(errors) * 100 if errors else float('inf')

    def ES_forecast(self, series, alpha):
        series = np.array(series)
        result = [series[0]]
        for n in range(1, len(series)):
            result.append(alpha * series[n] + (1 - alpha) * result[n-1])
        return result[-1]

    def ES_opt_alpha(self, series):
        series = np.array(series)
        alpha_list = np.linspace(0.1, 0.9, 100)
        errors = []
        
        for alpha in alpha_list:
            error = []
            for i in range(1, len(series)-1):
                forecast = self.ES_forecast(series[:i], alpha)
                error.append(abs(forecast - series[i+1]))
            errors.append(np.mean(error))
        
        return alpha_list[np.argmin(errors)]

    def evaluate_models(self, data, feature_columns, target_col='cantidad', 
                        lookback_periods=[6, 12, None], val_year=None):
        best_score = float('inf')
        best_config = None
        best_model = None

        data = data.sort_values(by='fecha')
        
        if len(data) < 4:
            return None, None, float('inf')
            
        # val_data = data[data['year'] == val_year] if val_year else data
        val_data = data
        val_size = 6

        for lookback in lookback_periods:
            if lookback:
                train_data = val_data.iloc[-(lookback+val_size):-val_size]
            else:
                train_data = val_data.iloc[:-val_size]
            test_data = val_data.iloc[-val_size:]

            # Imprimir datos de entrenamiento y prueba para este lookback
            print(f"\n--- Evaluando con lookback: {lookback} ---")
            print("Train Data:")
            print(train_data)
            print("Test Data:")
            print(test_data)
            
            y_train = train_data[target_col]
            y_test = test_data[target_col]
            
            models = {
                'mean': y_train.mean(),
                'median': y_train.median(),
                'es': (self.ES_forecast, self.ES_opt_alpha(y_train))
            }
            
            if len(feature_columns) > 0:
                models.update({
                    'xgboost': xgb.XGBRegressor(random_state=42),
                    'linear': LinearRegression()
                })
                X_train = train_data[feature_columns]
                X_test = test_data[feature_columns]
            
            for model_name, model in models.items():
                try:
                    if model_name in ['xgboost', 'linear']:
                        model.fit(X_train, y_train)
                        y_pred = model.predict(X_test)
                    elif model_name == 'es':
                        forecast_func, alpha = model
                        y_pred = []
                        current_data = y_train.values
                        for _ in range(len(y_test)):
                            pred = forecast_func(current_data, alpha)
                            y_pred.append(pred)
                            current_data = np.append(current_data, pred)
                    else:  # mean o median
                        y_pred = [model] * len(y_test)
                    
                    y_pred = np.maximum(y_pred, 0)
                    score = self.weighted_mape(y_test, y_pred)
                    
                    print(f"Modelo: {model_name}, Lookback: {lookback}, Score: {score}")
                    
                    if score < best_score:
                        best_score = score
                        best_config = (model_name, lookback)
                        best_model = model
                        
                except Exception as e:
                    print(f"Error evaluando el modelo {model_name} con lookback {lookback}: {e}")
                    continue
        
        return best_config, best_model, best_score

    def predict_future_months(self, data, feature_columns, best_model, 
                            best_model_name, lookback, num_months):
        predictions = []
        current_data = data.copy()
        current_data = current_data.sort_values(by="fecha")

        fecha_actual = current_data['fecha'].max()
        primer_dia_mes_actual = pd.Timestamp(fecha_actual.year, fecha_actual.month, 1)
        primer_dia_mes_anterior = primer_dia_mes_actual - pd.DateOffset(months=1)
        fecha_inicio = primer_dia_mes_anterior - pd.DateOffset(months=6)
        fecha_fin = primer_dia_mes_anterior + pd.offsets.MonthEnd(0)
        ultimos_seis_meses = current_data[
            (current_data['fecha'] >= fecha_inicio) & (current_data['fecha'] <= fecha_fin)
        ]

        for _ in range(num_months):
            if best_model_name in ['xgboost', 'linear']:
                pred = best_model.predict(current_data[feature_columns].iloc[-1:])[0]
            elif best_model_name in ['mean', 'median']:
                if not ultimos_seis_meses.empty:
                    pred = (ultimos_seis_meses['cantidad'].sum())/6 if best_model_name == 'mean' else ultimos_seis_meses['cantidad'].median()
                else:
                    pred = 0  # O puedes usar np.nan si prefieres manejarlo diferente
            else:  # ES
                forecast_func, alpha = best_model
                lookback_data = (current_data.iloc[-lookback:] if lookback 
                                else current_data)
                pred = forecast_func(lookback_data['cantidad'], alpha)
            
            pred = np.maximum(pred, 0)
            predictions.append(pred)
            
            new_row = current_data.iloc[-1:].copy()
            new_row['cantidad'] = pred
            new_row['fecha'] += pd.DateOffset(months=1)
            new_row['month'] = new_row['fecha'].dt.month
            new_row['year'] = new_row['fecha'].dt.year
            
            current_data = pd.concat([current_data, new_row])
            
            if feature_columns:
                for lag in range(1, 4):
                    current_data[f'cantidad_lag_{lag}'] = current_data['cantidad'].shift(lag)
        
        return predictions

    def make_final_predictions(self, all_monthly_data, df_cov, df_correlaciones_sig):
        results = []
        no_sku_process_list = []
        count = 0
        # for sku in all_monthly_data['articulo'].unique():
        for sku in ["paw68a500"]:
            print(f"Processing SKU: {sku}")
            try:
                sku_data = all_monthly_data[all_monthly_data['articulo'] == sku].copy()
                last_date = sku_data['fecha'].max()
                last_date = (last_date + pd.offsets.MonthBegin(1)).normalize()
                lt = int(sku_data['lt'].iloc[-1])
                last_year = sku_data['year'].max()

                # if len(sku_data) < 7:
                #     continue

                data, feature_columns = self.prepare_features_for_ml(
                    all_monthly_data, df_cov, df_correlaciones_sig, sku
                )

                best_config, best_model, score = self.evaluate_models(
                    data,
                    feature_columns,
                    lookback_periods=[6, 12, None],
                    val_year=last_year - 1
                )

                # if best_config is None:
                #     continue
                
                hoy = pd.Timestamp.today()
                primer_dia_mes_actual = pd.Timestamp(hoy.year, hoy.month, 1)
                primer_dia_mes_anterior = primer_dia_mes_actual - pd.DateOffset(months=1)
                fecha_inicio = primer_dia_mes_anterior - pd.DateOffset(months=6)
                fecha_fin = primer_dia_mes_anterior + pd.offsets.MonthEnd(0)  # último día del mes anterior
                ultimos_seis_meses = sku_data[
                    (sku_data['fecha'] >= fecha_inicio) & (sku_data['fecha'] <= fecha_fin)
                ]
                catusita = (ultimos_seis_meses['cantidad'].sum())/6

                if best_config is None or best_model is None:
                    # Ordenar por fecha por seguridad
                    sku_data = sku_data.sort_values(by='fecha')
                    
                    if not ultimos_seis_meses.empty:
                        avg_cantidad = (ultimos_seis_meses['cantidad'].sum())/6
                        caa = avg_cantidad
                        caa_lt = avg_cantidad
                        catusita = avg_cantidad
                        best_model_name = 'mean'
                    else:
                        caa = np.nan
                        caa_lt = np.nan
                        best_model_name = np.nan

                    results.append({
                        'sku': sku,
                        'lt': lt,
                        'date': last_date,
                        'model': best_model_name,
                        'real': 0,
                        'catusita': catusita,
                        'lookback_period': np.nan,
                        'features_used': 'none',
                        'caa': caa,
                        'caa_lt': caa_lt,
                        'corr_sd': np.nan,
                        'loss': np.nan
                    })

                    no_sku_process_list.append({'sku': sku})
                    count += 1
                    continue

                best_model_name, lookback = best_config

                # Calculate test score
                test_data = data[data['year'] == last_year]
                if feature_columns:
                    test_X = test_data[feature_columns]
                    if best_model_name in ['xgboost', 'linear']:
                        test_pred = best_model.predict(test_X)
                else:
                    if best_model_name in ['mean', 'median']:
                        test_pred = [best_model] * len(test_data)
                    else:  # ES
                        forecast_func, alpha = best_model
                        test_pred = []
                        current_data = data[data['year'] < last_year]['cantidad'].values
                        for _ in range(len(test_data)):
                            pred = forecast_func(current_data, alpha)
                            test_pred.append(pred)
                            current_data = np.append(current_data, pred)

                
                test_score = self.weighted_mape(test_data['cantidad'], test_pred)

                # Generate future predictions
                future_predictions = self.predict_future_months(
                    data,
                    feature_columns,
                    best_model,
                    best_model_name,
                    lookback,
                    2 * lt
                )

                # Calculate pred_std from the historical data used for prediction
                if lookback:
                    historical_data = data['cantidad'].iloc[-lookback:]
                else:
                    historical_data = data['cantidad']
                pred_std = np.std(historical_data)

                period1 = sum(future_predictions[:lt])
                period2 = sum(future_predictions[lt:2*lt])

                last_six_months_mean = (ultimos_seis_meses['cantidad'].sum())/6

                # Reemplazar predicción si es 0 y hay al menos un valor en ese rango
                if (
                    period1 == 0 and period2 == 0 and
                    ultimos_seis_meses['cantidad'].notna().sum() > 0
                ):
                    avg_cantidad = (ultimos_seis_meses['cantidad'].sum())/6
                    period1 = avg_cantidad
                    period2 = avg_cantidad
                    best_model_name = 'mean'

                # Registro final
                results.append({
                    'sku': sku,
                    'lt': lt,
                    'date': last_date,
                    'model': best_model_name,
                    'real': 0,
                    'catusita': last_six_months_mean,
                    'lookback_period': lookback if lookback else 'all',
                    'features_used': ','.join(feature_columns) if feature_columns else 'none',
                    'caa': period1,
                    'caa_lt': period2,
                    'corr_sd': pred_std,
                    'loss': test_score
                })

            except Exception as e:
                print(f"Error processing SKU {sku}: {str(e)}")
                continue
        print(f"El numero de SKU sin procesar: {count}")
        return pd.DataFrame(results)

    def process_predictions(self):
        from utils.process_data.config import DATA_PATHS
        
        data_path = DATA_PATHS['process'] / 'catusita_consolidated.csv'
        cov_path = DATA_PATHS['process'] / 'df_covariables.csv'
        
        all_monthly_data = self.create_monthly_sales_data(
            self.load_and_preprocess_data(data_path)
        )
        df_cov = self.load_covariates_data(cov_path)
        df_correlaciones_sig = pd.read_csv(DATA_PATHS['process'] / 'df_correlaciones_sig.csv')
        
        results_df = self.make_final_predictions(
            all_monthly_data, 
            df_cov, 
            df_correlaciones_sig
        )

        # results_df = results_df[results_df['date'] == results_df['date'].max()]

        if not results_df.empty:
            return results_df.sort_values('loss')
        return None


In [4]:
# Instanciar la clase Predictor
pred = Predictor()

# Definir la ruta al CSV (ajusta la ruta según corresponda)
data_path = 'data/process/catusita_consolidated.csv'

# Cargar y preprocesar los datos
df = pred.load_and_preprocess_data(data_path)

# Obtener el artículo más vendido en dólares
ventas_por_articulo = df.groupby('articulo')['venta_usd'].sum().reset_index()
ventas_por_articulo = ventas_por_articulo.sort_values('venta_usd', ascending=False)
articulo_mas_vendido = ventas_por_articulo.iloc[0]['articulo']
df = df[df['articulo'] == articulo_mas_vendido]

# Crear los datos mensuales de ventas
monthly_data = pred.create_monthly_sales_data(df)

# (Opcional) Imprimir las columnas para confirmar que el dataframe contiene lo esperado
print("Columnas de monthly_data:", monthly_data.columns)

# Definir las features que se usarán para el entrenamiento.
# Si no cuentas con features adicionales, puedes dejar la lista vacía para evaluar solo los modelos base.
feature_columns = []  # O, por ejemplo, ['transacciones', 'venta_pen'] si están disponibles

# Ejecutar la función evaluate_models y ver los prints de train_data, test_data y score de cada modelo
best_config, best_model, best_score = pred.evaluate_models(
    data=monthly_data,
    feature_columns=feature_columns,
    target_col='cantidad',
    lookback_periods=[6, 12, None]
)

print("\n--- Resultados finales ---")
print("Mejor configuración (modelo, lookback):", best_config)
print("Mejor modelo:", best_model)
print("Mejor score (weighted MAPE):", best_score)


Columnas de monthly_data: Index(['articulo', 'year', 'month', 'cantidad', 'transacciones', 'venta_pen',
       'fuente_suministro', 'lt', 'fecha'],
      dtype='object')

--- Evaluando con lookback: 6 ---
Train Data:
     articulo  year  month  cantidad  transacciones  venta_pen  \
39  paw68a500  2024      4     456.0             28   57327.50   
40  paw68a500  2024      5     719.0             30   90082.59   
41  paw68a500  2024      6     473.0             25   59556.25   
42  paw68a500  2024      7     674.0             35   84600.35   
43  paw68a500  2024      8     799.0             39   96249.06   
44  paw68a500  2024      9     837.0             36  101765.23   

           fuente_suministro   lt      fecha  
39  prextoline - lubricantes  2.0 2024-04-01  
40  prextoline - lubricantes  2.0 2024-05-01  
41  prextoline - lubricantes  2.0 2024-06-01  
42  prextoline - lubricantes  2.0 2024-07-01  
43  prextoline - lubricantes  2.0 2024-08-01  
44  prextoline - lubricantes  2.0 2024